# Step 4 - Global Direction Discovery

This notebook implements Step 4 for global latent directions using Step 3 artifacts only.


## (0) Constants + paths

Set one place for constants and file paths (single-seed, no config files).

In [ ]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

SEED = 42
RIDGE_ALPHA_GRID = np.logspace(-3, 3, 13)
B = 8
RANDOM_K = 200

A_R2_TEST_MIN = 0.50
A_STABILITY_MIN = 0.85
A_MAX_CONFOUND_COS_MAX = 0.30
B_R2_TEST_MIN = 0.20
B_STABILITY_MIN = 0.65
CONFOUND_WARNING_THRESHOLD = 0.50

MIN_TRAIN_POINTS = 25
MIN_EVAL_POINTS = 10
RESID_PREFIX = "resid_"

CONFOUND_COLS = [
    "selfies_len_tokens",
    "ring_token_count",
    "branch_token_count",
    "token_entropy",
]

Y_COLS_EXPECTED = [
    "MolWt", "ExactMolWt", "HeavyAtomCount", "cLogP", "TPSA",
    "HBD", "HBA", "NumRotatableBonds", "RingCount", "AromaticRingCount",
    "FractionCSP3", "NumSpiroAtoms", "NumBridgeheadAtoms", "BertzCT", "QED",
]

cwd = Path.cwd().resolve()
if (cwd / "artifacts").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "artifacts").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

STEP3_DIR = PROJECT_ROOT / "artifacts" / "confounds_step3"
STEP3_TABLES_DIR = STEP3_DIR / "tables"
STEP3_CONFOUND_DIR = STEP3_DIR / "confound_directions"
STEP3_NOTEBOOK_CANDIDATES = [
    PROJECT_ROOT / "transformer_vae" / "step3_confounds_rdkit_properties.ipynb",
    PROJECT_ROOT / "transformer_vae" / "step3_confounds_rdkit_properties_gpu.ipynb",
    cwd / "step3_confounds_rdkit_properties.ipynb",
]

Z_PATH = STEP3_DIR / "Z_mu.npy"
PANELS_PATH = STEP3_DIR / "panels_step3_with_residuals.csv"
R2_Y_PATH = STEP3_TABLES_DIR / "r2_Z_to_Y.csv"

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "directions_step4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALERS_PATH = OUTPUT_DIR / "scalers.npz"
DIRECTIONS_RAW_PATH = OUTPUT_DIR / "directions_raw.npz"
DIRECTIONS_RESID_PATH = OUTPUT_DIR / "directions_resid.npz"
CONTROLS_PERMUTED_PATH = OUTPUT_DIR / "controls_permuted.npz"
CONTROLS_RANDOM_PATH = OUTPUT_DIR / "controls_random.npz"

R2_RAW_CSV = OUTPUT_DIR / "r2_raw.csv"
R2_RESID_CSV = OUTPUT_DIR / "r2_resid.csv"
STAB_RAW_CSV = OUTPUT_DIR / "stability_bootstrap_raw.csv"
STAB_RESID_CSV = OUTPUT_DIR / "stability_bootstrap_resid.csv"
COS_RAW_CSV = OUTPUT_DIR / "cosine_vs_confounds_raw.csv"
COS_RESID_CSV = OUTPUT_DIR / "cosine_vs_confounds_resid.csv"
LABELS_CSV = OUTPUT_DIR / "direction_labels_ABC.csv"
SUMMARY_JSON = OUTPUT_DIR / "summary_step4.json"

PLOT_R2_COMPARISON = OUTPUT_DIR / "r2_comparison_raw_vs_resid.png"
PLOT_STABILITY = OUTPUT_DIR / "stability_summary.png"
PLOT_COS_HEATMAP = OUTPUT_DIR / "cosine_vs_confounds_heatmaps.png"
PLOT_CONTROLS = OUTPUT_DIR / "controls_summary.png"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STEP3_DIR:", STEP3_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## (1) Inspect Step 3 to detect what is already done

Load Step 3 artifacts and notebook code to detect prior scaling/alpha behavior.

Decision rule:
- reuse Step 3 scalers only if reusable scaler artifacts are present
- otherwise do train-only standardization in Step 4 and save `scalers.npz`

In [ ]:
assert Z_PATH.exists(), f"Missing {Z_PATH}"
assert PANELS_PATH.exists(), f"Missing {PANELS_PATH}"

Z_mu = np.load(Z_PATH).astype(np.float32)
panels = pd.read_csv(PANELS_PATH)

print("Z_mu shape:", Z_mu.shape, "dtype:", Z_mu.dtype)
print("panels shape:", panels.shape)
print("split counts:\n", panels["split"].value_counts(dropna=False))

step3_notebook_path = next((p for p in STEP3_NOTEBOOK_CANDIDATES if p.exists()), None)

scale_tokens = [
    "StandardScaler", "sklearn.preprocessing", "z-score", "zscore", "mean", "std", "fit_transform"
]
step3_scale_evidence = {"notebook_found": step3_notebook_path is not None, "token_hits": {}}

if step3_notebook_path is not None:
    nb3 = json.loads(step3_notebook_path.read_text(encoding="utf-8"))
    code_text = "\n".join(
        "".join(c.get("source", []))
        for c in nb3.get("cells", [])
        if c.get("cell_type") == "code"
    )
    for tok in scale_tokens:
        step3_scale_evidence["token_hits"][tok] = bool(re.search(re.escape(tok), code_text, flags=re.IGNORECASE))

saved_scaler_candidates = [
    STEP3_DIR / "scalers.npz",
    STEP3_DIR / "z_scaler.npz",
    STEP3_DIR / "standardization.npz",
    STEP3_TABLES_DIR / "scalers.npz",
]
step3_saved_scalers_found = any(x.exists() for x in saved_scaler_candidates)
step3_mentions_scaling = any(step3_scale_evidence["token_hits"].values()) if step3_scale_evidence["token_hits"] else False

alpha_map_step3 = {}
if R2_Y_PATH.exists():
    _r2y = pd.read_csv(R2_Y_PATH)
    if {"property", "alpha"}.issubset(_r2y.columns):
        for _, row in _r2y.iterrows():
            a = float(row["alpha"])
            if np.isfinite(a) and a > 0:
                alpha_map_step3[str(row["property"])] = a

PERFORM_STANDARDIZATION_STEP4 = not step3_saved_scalers_found
if PERFORM_STANDARDIZATION_STEP4:
    standardization_note = (
        "Step 3 used scaling inside fits, but no reusable scaler artifact was found. "
        "Step 4 will standardize on train only and save scalers."
    )
else:
    standardization_note = "Reusable Step 3 scalers found; Step 4 will not re-standardize."

standardization_decision = {
    "step3_notebook_path": str(step3_notebook_path) if step3_notebook_path else None,
    "step3_mentions_scaling": bool(step3_mentions_scaling),
    "step3_saved_scalers_found": bool(step3_saved_scalers_found),
    "perform_standardization_step4": bool(PERFORM_STANDARDIZATION_STEP4),
    "note": standardization_note,
}

print("Step 3 notebook:", step3_notebook_path)
print("Step 3 scale evidence:", step3_scale_evidence)
print("Loaded Step 3 alpha count:", len(alpha_map_step3))
print("Decision:", standardization_note)


## (2) Data extraction

Identify raw properties `Y`, residual properties `Y_resid`, confounds `C`, and verify row alignment with `Z_mu`.

In [ ]:
assert "split" in panels.columns, "`split` column is required"

missing_c = [c for c in CONFOUND_COLS if c not in panels.columns]
assert not missing_c, f"Missing confound columns: {missing_c}"

Y_COLS = [c for c in Y_COLS_EXPECTED if c in panels.columns]
assert len(Y_COLS) == 15, f"Expected 15 Y columns, found {len(Y_COLS)}"

Y_RESID_COLS = [f"{RESID_PREFIX}{c}" for c in Y_COLS if f"{RESID_PREFIX}{c}" in panels.columns]
assert len(Y_RESID_COLS) == len(Y_COLS), "Missing residual columns"

N_z = int(Z_mu.shape[0])
N_panel = int(len(panels))
assert N_z == N_panel, f"Row mismatch: Z={N_z}, panels={N_panel}"

id_candidates = [c for c in panels.columns if c.lower() in {"id", "idx", "row_id", "index"}]
if id_candidates:
    id_col = id_candidates[0]
    if panels[id_col].is_unique:
        print(f"Alignment check via unique ID column: {id_col}")
    else:
        warnings.warn(f"ID column `{id_col}` not unique; using row order only.")
else:
    warnings.warn("No explicit ID/index column; using row-order alignment because N matches.")

split_series = panels["split"].astype(str).str.lower().str.strip()
split_masks = {k: (split_series == k).to_numpy() for k in ["train", "val", "test"]}
for k in ["train", "val", "test"]:
    assert split_masks[k].sum() > 0, f"No rows in split `{k}`"

Y_raw_all = panels[Y_COLS].to_numpy(dtype=np.float64)
Y_resid_all = panels[Y_RESID_COLS].to_numpy(dtype=np.float64)

print("Y columns:", Y_COLS)
print("Y_resid columns:", Y_RESID_COLS)
print("split sizes:", {k: int(v.sum()) for k, v in split_masks.items()})

## (3) Standardization (if needed)

Fit train-only means/stds for `Z` and targets, transform splits when needed, then save `scalers.npz`.

In [ ]:
def compute_target_train_stats(Y, train_mask):
    means = np.full(Y.shape[1], np.nan, dtype=np.float64)
    stds = np.full(Y.shape[1], np.nan, dtype=np.float64)
    for j in range(Y.shape[1]):
        ytr = Y[train_mask, j]
        finite = np.isfinite(ytr)
        if finite.sum() < 2:
            continue
        means[j] = float(np.mean(ytr[finite]))
        s = float(np.std(ytr[finite]))
        stds[j] = 1.0 if s < 1e-8 else s
    return means, stds

train_mask = split_masks["train"]

z_train = Z_mu[train_mask]
z_mean = z_train.mean(axis=0).astype(np.float64)
z_std = z_train.std(axis=0).astype(np.float64)
z_std[z_std < 1e-8] = 1.0

y_raw_mean, y_raw_std = compute_target_train_stats(Y_raw_all, train_mask)
y_resid_mean, y_resid_std = compute_target_train_stats(Y_resid_all, train_mask)

if PERFORM_STANDARDIZATION_STEP4:
    Z_fit_all = ((Z_mu - z_mean) / z_std).astype(np.float32)
else:
    Z_fit_all = Z_mu.astype(np.float32)

np.savez(
    SCALERS_PATH,
    z_mean=z_mean.astype(np.float32),
    z_std=z_std.astype(np.float32),
    y_raw_mean=y_raw_mean,
    y_raw_std=y_raw_std,
    y_resid_mean=y_resid_mean,
    y_resid_std=y_resid_std,
    y_raw_names=np.array(Y_COLS),
    y_resid_names=np.array(Y_RESID_COLS),
    standardize_in_step4=np.array([PERFORM_STANDARDIZATION_STEP4], dtype=bool),
    step3_mentions_scaling=np.array([standardization_decision["step3_mentions_scaling"]], dtype=bool),
    step3_saved_scalers_found=np.array([standardization_decision["step3_saved_scalers_found"]], dtype=bool),
)

print("Saved:", SCALERS_PATH)

## (4) Fit GLOBAL directions (raw `Y`)

For each property: reuse Step 3 alpha if available, otherwise choose alpha on train->val; then fit ridge and save directions.

In [ ]:
def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    finite = np.isfinite(y_true) & np.isfinite(y_pred)
    if finite.sum() < MIN_EVAL_POINTS:
        return np.nan
    yt, yp = y_true[finite], y_pred[finite]
    if np.std(yt) < 1e-12:
        return np.nan
    return float(r2_score(yt, yp))


def cosine(u, v):
    u = np.asarray(u, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    nu = np.linalg.norm(u)
    nv = np.linalg.norm(v)
    if nu < 1e-12 or nv < 1e-12:
        return np.nan
    return float(np.dot(u, v) / (nu * nv))


def unit(v):
    v = np.asarray(v, dtype=np.float64)
    n = np.linalg.norm(v)
    if n < 1e-12:
        return np.zeros_like(v)
    return v / n


def coef_to_original_space(coef_fit, intercept_fit, y_mean, y_std):
    coef_fit = np.asarray(coef_fit, dtype=np.float64)
    intercept_fit = float(intercept_fit)
    if PERFORM_STANDARDIZATION_STEP4:
        w = (y_std * coef_fit) / z_std
        b = y_mean + y_std * (intercept_fit - float(np.dot(z_mean / z_std, coef_fit)))
    else:
        w = coef_fit
        b = intercept_fit
    return w.astype(np.float64), float(b)


def choose_alpha_on_val(Xtr, ytr_fit, Xva, yva_orig, y_mean, y_std):
    best_alpha = float(RIDGE_ALPHA_GRID[0])
    best_r2 = -np.inf
    for alpha in RIDGE_ALPHA_GRID:
        model = Ridge(alpha=float(alpha), fit_intercept=True)
        model.fit(Xtr, ytr_fit)
        p = model.predict(Xva)
        if PERFORM_STANDARDIZATION_STEP4:
            p = p * y_std + y_mean
        r2 = safe_r2(yva_orig, p)
        if np.isfinite(r2) and r2 > best_r2:
            best_r2 = r2
            best_alpha = float(alpha)
    return best_alpha


def fit_direction_set(target_matrix, target_names, y_means, y_stds, alpha_overrides, out_npz, out_csv):
    rows, w_list, wu_list, b_list, a_list, a_src = [], [], [], [], [], []
    for j, target in enumerate(target_names):
        y_all = target_matrix[:, j]
        tr = split_masks["train"] & np.isfinite(y_all)
        va = split_masks["val"] & np.isfinite(y_all)
        te = split_masks["test"] & np.isfinite(y_all)

        ntr, nva, nte = int(tr.sum()), int(va.sum()), int(te.sum())
        y_mean = float(y_means[j]) if np.isfinite(y_means[j]) else 0.0
        y_std = float(y_stds[j]) if np.isfinite(y_stds[j]) and y_stds[j] > 0 else 1.0

        if ntr < MIN_TRAIN_POINTS:
            z = np.zeros(Z_mu.shape[1], dtype=np.float64)
            rows.append({"target":target,"alpha":np.nan,"alpha_source":"insufficient_train","n_train":ntr,"n_val":nva,"n_test":nte,"r2_train":np.nan,"r2_val":np.nan,"r2_test":np.nan,"w_norm":0.0})
            w_list.append(z); wu_list.append(z); b_list.append(np.nan); a_list.append(np.nan); a_src.append("insufficient_train")
            continue

        Xtr, Xva, Xte = Z_fit_all[tr], Z_fit_all[va], Z_fit_all[te]
        ytr, yva, yte = y_all[tr].astype(np.float64), y_all[va].astype(np.float64), y_all[te].astype(np.float64)
        ytr_fit = (ytr - y_mean) / y_std if PERFORM_STANDARDIZATION_STEP4 else ytr

        a_override = alpha_overrides.get(target, np.nan)
        if np.isfinite(a_override) and a_override > 0:
            alpha, alpha_source = float(a_override), "step3_r2_Z_to_Y"
        else:
            alpha, alpha_source = choose_alpha_on_val(Xtr, ytr_fit, Xva, yva, y_mean, y_std), "step4_val_search"

        model = Ridge(alpha=alpha, fit_intercept=True)
        model.fit(Xtr, ytr_fit)

        ptr, pva, pte = model.predict(Xtr), model.predict(Xva), model.predict(Xte)
        if PERFORM_STANDARDIZATION_STEP4:
            ptr = ptr * y_std + y_mean
            pva = pva * y_std + y_mean
            pte = pte * y_std + y_mean

        w, b = coef_to_original_space(model.coef_, model.intercept_, y_mean, y_std)
        rows.append({"target":target,"alpha":alpha,"alpha_source":alpha_source,"n_train":ntr,"n_val":nva,"n_test":nte,"r2_train":safe_r2(ytr, ptr),"r2_val":safe_r2(yva, pva),"r2_test":safe_r2(yte, pte),"w_norm":float(np.linalg.norm(w))})
        w_list.append(w); wu_list.append(unit(w)); b_list.append(b); a_list.append(alpha); a_src.append(alpha_source)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    w_arr = np.vstack(w_list).astype(np.float32)
    wu_arr = np.vstack(wu_list).astype(np.float32)
    b_arr = np.array(b_list, dtype=np.float64)
    a_arr = np.array(a_list, dtype=np.float64)

    np.savez(
        out_npz,
        target_names=np.array(target_names),
        w=w_arr,
        w_unit=wu_arr,
        intercept=b_arr,
        alpha=a_arr,
        alpha_source=np.array(a_src),
    )

    return df, w_arr, wu_arr, b_arr, a_arr

In [ ]:
alpha_overrides_raw = {name: alpha_map_step3.get(name, np.nan) for name in Y_COLS}

r2_raw_df, w_raw, w_raw_unit, b_raw, alpha_raw = fit_direction_set(
    target_matrix=Y_raw_all,
    target_names=Y_COLS,
    y_means=y_raw_mean,
    y_stds=y_raw_std,
    alpha_overrides=alpha_overrides_raw,
    out_npz=DIRECTIONS_RAW_PATH,
    out_csv=R2_RAW_CSV,
)

print("Saved:", DIRECTIONS_RAW_PATH)
print("Saved:", R2_RAW_CSV)

## (5) Fit GLOBAL directions (residual `Y_resid`)

Repeat Step 4 for residual targets and save residual direction artifacts.

In [ ]:
alpha_overrides_resid = {
    resid_name: alpha_map_step3.get(resid_name[len(RESID_PREFIX):], np.nan)
    for resid_name in Y_RESID_COLS
}

r2_resid_df, w_resid, w_resid_unit, b_resid, alpha_resid = fit_direction_set(
    target_matrix=Y_resid_all,
    target_names=Y_RESID_COLS,
    y_means=y_resid_mean,
    y_stds=y_resid_std,
    alpha_overrides=alpha_overrides_resid,
    out_npz=DIRECTIONS_RESID_PATH,
    out_csv=R2_RESID_CSV,
)

raw_plot = r2_raw_df[["target", "r2_test"]].copy()
raw_plot["property"] = raw_plot["target"]
raw_plot = raw_plot.set_index("property")["r2_test"]

resid_plot = r2_resid_df[["target", "r2_test"]].copy()
resid_plot["property"] = resid_plot["target"].str.replace(f"^{RESID_PREFIX}", "", regex=True)
resid_plot = resid_plot.set_index("property")["r2_test"]

order = [p for p in Y_COLS if p in raw_plot.index and p in resid_plot.index]
x = np.arange(len(order))
w = 0.38

plt.figure(figsize=(14, 6))
plt.bar(x - w / 2, raw_plot.loc[order].to_numpy(), width=w, label="raw")
plt.bar(x + w / 2, resid_plot.loc[order].to_numpy(), width=w, label="resid")
plt.axhline(0.0, color="black", linewidth=1)
plt.xticks(x, order, rotation=80, ha="right")
plt.ylabel("Test R2")
plt.title("Step 4: Test R2 (raw vs residual)")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_R2_COMPARISON, dpi=220)
plt.close()

print("Saved:", DIRECTIONS_RESID_PATH)
print("Saved:", R2_RESID_CSV)
print("Saved:", PLOT_R2_COMPARISON)

## (6) Direction stability via bootstrap

For each direction, run `B` train bootstrap fits at fixed alpha, sign-align to full direction, then summarize cosine stability.

In [ ]:
def bootstrap_stability(target_matrix, target_names, y_means, y_stds, w_full, alpha_used, out_csv, seed_offset):
    rng = np.random.default_rng(SEED + seed_offset)
    rows = []

    for j, target in enumerate(target_names):
        y_all = target_matrix[:, j]
        tr = split_masks["train"] & np.isfinite(y_all)
        ntr = int(tr.sum())

        alpha = float(alpha_used[j]) if np.isfinite(alpha_used[j]) else np.nan
        y_mean = float(y_means[j]) if np.isfinite(y_means[j]) else 0.0
        y_std = float(y_stds[j]) if np.isfinite(y_stds[j]) and y_stds[j] > 0 else 1.0
        full_w = w_full[j].astype(np.float64)

        if (not np.isfinite(alpha)) or ntr < MIN_TRAIN_POINTS or np.linalg.norm(full_w) < 1e-12:
            rows.append({"target":target,"alpha":alpha,"n_train":ntr,"B":B,"cos_median":np.nan,"cos_mean":np.nan,"cos_p05":np.nan,"cos_p95":np.nan})
            continue

        Xtr = Z_fit_all[tr]
        ytr = y_all[tr].astype(np.float64)
        ytr_fit = (ytr - y_mean) / y_std if PERFORM_STANDARDIZATION_STEP4 else ytr

        cos_vals = []
        n = Xtr.shape[0]
        for _ in range(B):
            idx = rng.integers(0, n, size=n)
            model = Ridge(alpha=alpha, fit_intercept=True)
            model.fit(Xtr[idx], ytr_fit[idx])
            wb, _ = coef_to_original_space(model.coef_, model.intercept_, y_mean, y_std)
            c = cosine(wb, full_w)
            if np.isfinite(c):
                cos_vals.append(abs(c))

        arr = np.array(cos_vals if len(cos_vals) else [np.nan], dtype=np.float64)
        rows.append({
            "target":target,"alpha":alpha,"n_train":ntr,"B":B,
            "cos_median":float(np.nanmedian(arr)),
            "cos_mean":float(np.nanmean(arr)),
            "cos_p05":float(np.nanpercentile(arr, 5)),
            "cos_p95":float(np.nanpercentile(arr, 95)),
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

stability_raw_df = bootstrap_stability(
    target_matrix=Y_raw_all,
    target_names=Y_COLS,
    y_means=y_raw_mean,
    y_stds=y_raw_std,
    w_full=w_raw,
    alpha_used=alpha_raw,
    out_csv=STAB_RAW_CSV,
    seed_offset=1000,
)

stability_resid_df = bootstrap_stability(
    target_matrix=Y_resid_all,
    target_names=Y_RESID_COLS,
    y_means=y_resid_mean,
    y_stds=y_resid_std,
    w_full=w_resid,
    alpha_used=alpha_resid,
    out_csv=STAB_RESID_CSV,
    seed_offset=2000,
)

st_raw = stability_raw_df[["target", "cos_median"]].copy()
st_raw["property"] = st_raw["target"]
st_raw = st_raw.set_index("property")["cos_median"]

st_res = stability_resid_df[["target", "cos_median"]].copy()
st_res["property"] = st_res["target"].str.replace(f"^{RESID_PREFIX}", "", regex=True)
st_res = st_res.set_index("property")["cos_median"]

order = [p for p in Y_COLS if p in st_raw.index and p in st_res.index]
x = np.arange(len(order))
w = 0.38

plt.figure(figsize=(14, 6))
plt.bar(x - w / 2, st_raw.loc[order].to_numpy(), width=w, label="raw")
plt.bar(x + w / 2, st_res.loc[order].to_numpy(), width=w, label="resid")
plt.ylim(0, 1.02)
plt.xticks(x, order, rotation=80, ha="right")
plt.ylabel("Median bootstrap cosine")
plt.title("Step 4: Stability summary")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_STABILITY, dpi=220)
plt.close()

print("Saved:", STAB_RAW_CSV)
print("Saved:", STAB_RESID_CSV)
print("Saved:", PLOT_STABILITY)

## (7) Confound alignment diagnostics

Load Step 3 confound directions and compute cosine alignment with each Step 4 property direction.

In [ ]:
confound_files = {
    "selfies_len_tokens": STEP3_CONFOUND_DIR / "w_length.npy",
    "ring_token_count": STEP3_CONFOUND_DIR / "w_ring.npy",
    "branch_token_count": STEP3_CONFOUND_DIR / "w_branch.npy",
    "token_entropy": STEP3_CONFOUND_DIR / "w_entropy.npy",
}
for k, v in confound_files.items():
    assert v.exists(), f"Missing confound direction: {k} -> {v}"

confound_names = list(confound_files.keys())
confound_w = np.vstack([np.load(confound_files[n]).astype(np.float64) for n in confound_names])
confound_w_unit = np.vstack([unit(v) for v in confound_w])

def cosine_table(target_names, w_unit_matrix, out_csv):
    rows = []
    for i, t in enumerate(target_names):
        row = {"target": t}
        vals = []
        for j, cname in enumerate(confound_names):
            c = cosine(w_unit_matrix[i], confound_w_unit[j])
            row[cname] = c
            vals.append(abs(c) if np.isfinite(c) else np.nan)
        row["max_abs_confound_cos"] = float(np.nanmax(vals)) if np.any(np.isfinite(vals)) else np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

cosine_raw_df = cosine_table(Y_COLS, w_raw_unit, COS_RAW_CSV)
cosine_resid_df = cosine_table(Y_RESID_COLS, w_resid_unit, COS_RESID_CSV)

mat_raw = cosine_raw_df[confound_names].to_numpy(dtype=float)
mat_res = cosine_resid_df[confound_names].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=False)
im0 = axes[0].imshow(mat_raw, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set_title("Raw vs confounds")
axes[0].set_xticks(np.arange(len(confound_names)))
axes[0].set_xticklabels(confound_names, rotation=45, ha="right")
axes[0].set_yticks(np.arange(len(Y_COLS)))
axes[0].set_yticklabels(Y_COLS)

im1 = axes[1].imshow(mat_res, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("Residual vs confounds")
axes[1].set_xticks(np.arange(len(confound_names)))
axes[1].set_xticklabels(confound_names, rotation=45, ha="right")
axes[1].set_yticks(np.arange(len(Y_RESID_COLS)))
axes[1].set_yticklabels(Y_RESID_COLS)

fig.colorbar(im1, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02, label="cosine")
fig.tight_layout()
fig.savefig(PLOT_COS_HEATMAP, dpi=220)
plt.close(fig)

print("Saved:", COS_RAW_CSV)
print("Saved:", COS_RESID_CSV)
print("Saved:", PLOT_COS_HEATMAP)

## (8) Controls

Run:
- permuted-label control (per target)
- random unit-direction control (seeded)

In [ ]:
rng_perm = np.random.default_rng(SEED + 3000)
perm_rows = []

specs = [
    ("raw", Y_COLS, Y_raw_all, y_raw_mean, y_raw_std, alpha_raw),
    ("resid", Y_RESID_COLS, Y_resid_all, y_resid_mean, y_resid_std, alpha_resid),
]

for target_type, target_names, target_matrix, y_means, y_stds, alpha_used in specs:
    for j, target in enumerate(target_names):
        y_all = target_matrix[:, j]
        tr = split_masks["train"] & np.isfinite(y_all)
        va = split_masks["val"] & np.isfinite(y_all)
        te = split_masks["test"] & np.isfinite(y_all)

        alpha = float(alpha_used[j]) if np.isfinite(alpha_used[j]) else np.nan
        y_mean = float(y_means[j]) if np.isfinite(y_means[j]) else 0.0
        y_std = float(y_stds[j]) if np.isfinite(y_stds[j]) and y_stds[j] > 0 else 1.0

        if (not np.isfinite(alpha)) or tr.sum() < MIN_TRAIN_POINTS:
            perm_rows.append({"target_type":target_type,"target":target,"alpha":alpha,"r2_val_permuted":np.nan,"r2_test_permuted":np.nan,"w_norm_permuted":np.nan})
            continue

        Xtr, Xva, Xte = Z_fit_all[tr], Z_fit_all[va], Z_fit_all[te]
        ytr, yva, yte = y_all[tr].astype(np.float64), y_all[va].astype(np.float64), y_all[te].astype(np.float64)
        ytr_perm = ytr[rng_perm.permutation(len(ytr))]
        ytr_perm_fit = (ytr_perm - y_mean) / y_std if PERFORM_STANDARDIZATION_STEP4 else ytr_perm

        model = Ridge(alpha=alpha, fit_intercept=True)
        model.fit(Xtr, ytr_perm_fit)

        pva, pte = model.predict(Xva), model.predict(Xte)
        if PERFORM_STANDARDIZATION_STEP4:
            pva = pva * y_std + y_mean
            pte = pte * y_std + y_mean

        w_perm, _ = coef_to_original_space(model.coef_, model.intercept_, y_mean, y_std)
        perm_rows.append({
            "target_type":target_type,
            "target":target,
            "alpha":alpha,
            "r2_val_permuted":safe_r2(yva, pva),
            "r2_test_permuted":safe_r2(yte, pte),
            "w_norm_permuted":float(np.linalg.norm(w_perm)),
        })

permuted_df = pd.DataFrame(perm_rows)
np.savez(
    CONTROLS_PERMUTED_PATH,
    target_type=permuted_df["target_type"].to_numpy(),
    target=permuted_df["target"].to_numpy(),
    alpha=permuted_df["alpha"].to_numpy(dtype=np.float64),
    r2_val_permuted=permuted_df["r2_val_permuted"].to_numpy(dtype=np.float64),
    r2_test_permuted=permuted_df["r2_test_permuted"].to_numpy(dtype=np.float64),
    w_norm_permuted=permuted_df["w_norm_permuted"].to_numpy(dtype=np.float64),
)

rng_rand = np.random.default_rng(SEED + 4000)
random_dirs = rng_rand.normal(size=(RANDOM_K, Z_mu.shape[1])).astype(np.float64)
random_dirs = random_dirs / np.clip(np.linalg.norm(random_dirs, axis=1, keepdims=True), 1e-12, None)

rand_cos_conf = random_dirs @ confound_w_unit.T
rand_max_abs_conf = np.max(np.abs(rand_cos_conf), axis=1)

np.savez(
    CONTROLS_RANDOM_PATH,
    random_unit_dirs=random_dirs.astype(np.float32),
    cosine_vs_confounds=rand_cos_conf.astype(np.float32),
    max_abs_confound_cos=rand_max_abs_conf.astype(np.float32),
)

true_r2 = pd.concat([
    r2_raw_df.assign(target_type="raw")[["target_type", "target", "r2_test"]],
    r2_resid_df.assign(target_type="resid")[["target_type", "target", "r2_test"]],
], ignore_index=True)

plot_df = true_r2.merge(
    permuted_df[["target_type", "target", "r2_test_permuted"]],
    on=["target_type", "target"],
    how="left",
)

raw_max = cosine_raw_df["max_abs_confound_cos"].to_numpy(dtype=float)
resid_max = cosine_resid_df["max_abs_confound_cos"].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ttype, marker, color in [("raw", "o", "tab:blue"), ("resid", "^", "tab:orange")]:
    d = plot_df[plot_df["target_type"] == ttype]
    axes[0].scatter(d["r2_test_permuted"], d["r2_test"], marker=marker, color=color, alpha=0.8, label=ttype)
axes[0].axvline(0.0, color="black", linewidth=1)
axes[0].axhline(0.0, color="black", linewidth=1)
axes[0].set_xlabel("Permuted test R2")
axes[0].set_ylabel("True test R2")
axes[0].set_title("Permutation control")
axes[0].legend()

axes[1].hist(rand_max_abs_conf, bins=30, alpha=0.75, color="gray", label="random")
axes[1].axvline(np.nanmedian(raw_max), color="tab:blue", linestyle="--", label="raw median")
axes[1].axvline(np.nanmedian(resid_max), color="tab:orange", linestyle="--", label="resid median")
axes[1].set_xlabel("Max abs cosine vs confounds")
axes[1].set_ylabel("Count")
axes[1].set_title("Random direction control")
axes[1].legend()

fig.tight_layout()
fig.savefig(PLOT_CONTROLS, dpi=220)
plt.close(fig)

print("Saved:", CONTROLS_PERMUTED_PATH)
print("Saved:", CONTROLS_RANDOM_PATH)
print("Saved:", PLOT_CONTROLS)

## (9) Label directions A/B/C

Heuristic:
- `A`: high test R2 + high stability + low confound alignment
- `B`: moderate R2 or moderate stability
- `C`: unstable or near-zero predictive power

In [ ]:
def build_label_table(r2_df, stab_df, cos_df, target_type):
    d = r2_df[["target", "r2_test"]].merge(
        stab_df[["target", "cos_median"]], on="target", how="left"
    ).merge(
        cos_df[["target", "max_abs_confound_cos"]], on="target", how="left"
    )
    d["target_type"] = target_type
    return d


def assign_label(row):
    r2, stab, conf = row["r2_test"], row["cos_median"], row["max_abs_confound_cos"]
    if np.isfinite(r2) and np.isfinite(stab) and np.isfinite(conf):
        if (r2 >= A_R2_TEST_MIN) and (stab >= A_STABILITY_MIN) and (conf <= A_MAX_CONFOUND_COS_MAX):
            return "A"
        if (r2 >= B_R2_TEST_MIN) or (stab >= B_STABILITY_MIN):
            return "B"
    return "C"

label_raw = build_label_table(r2_raw_df, stability_raw_df, cosine_raw_df, "raw")
label_resid = build_label_table(r2_resid_df, stability_resid_df, cosine_resid_df, "resid")
labels_df = pd.concat([label_raw, label_resid], ignore_index=True)
labels_df["label"] = labels_df.apply(assign_label, axis=1)

labels_df = labels_df[[
    "target_type", "target", "label", "r2_test", "cos_median", "max_abs_confound_cos"
]]
labels_df.to_csv(LABELS_CSV, index=False)

print("Saved:", LABELS_CSV)
print(labels_df.head(10))

## (10) Save artifacts + final summary

Save concise JSON summary with top directions and confound-alignment warnings.

In [ ]:
summary_raw = labels_df[labels_df["target_type"] == "raw"].copy()
summary_resid = labels_df[labels_df["target_type"] == "resid"].copy()

raw_top = (
    summary_raw.sort_values(["r2_test", "cos_median"], ascending=[False, False])
    .head(5)[["target", "label", "r2_test", "cos_median", "max_abs_confound_cos"]]
    .to_dict(orient="records")
)
resid_top = (
    summary_resid.sort_values(["r2_test", "cos_median"], ascending=[False, False])
    .head(5)[["target", "label", "r2_test", "cos_median", "max_abs_confound_cos"]]
    .to_dict(orient="records")
)

warning_df = labels_df[labels_df["max_abs_confound_cos"] >= CONFOUND_WARNING_THRESHOLD].copy()
warning_df = warning_df.sort_values("max_abs_confound_cos", ascending=False)
warnings_top = warning_df[[
    "target_type", "target", "label", "max_abs_confound_cos", "r2_test", "cos_median"
]].head(10).to_dict(orient="records")

summary = {
    "standardization": standardization_decision,
    "top5_raw_directions": raw_top,
    "top5_resid_directions": resid_top,
    "confound_alignment_warnings": warnings_top,
    "artifacts": {
        "scalers": str(SCALERS_PATH),
        "directions_raw": str(DIRECTIONS_RAW_PATH),
        "directions_resid": str(DIRECTIONS_RESID_PATH),
        "controls_permuted": str(CONTROLS_PERMUTED_PATH),
        "controls_random": str(CONTROLS_RANDOM_PATH),
        "r2_raw_csv": str(R2_RAW_CSV),
        "r2_resid_csv": str(R2_RESID_CSV),
        "stability_raw_csv": str(STAB_RAW_CSV),
        "stability_resid_csv": str(STAB_RESID_CSV),
        "cosine_raw_csv": str(COS_RAW_CSV),
        "cosine_resid_csv": str(COS_RESID_CSV),
        "labels_csv": str(LABELS_CSV),
        "plot_r2_comparison": str(PLOT_R2_COMPARISON),
        "plot_stability": str(PLOT_STABILITY),
        "plot_cos_heatmap": str(PLOT_COS_HEATMAP),
        "plot_controls": str(PLOT_CONTROLS),
    },
}

SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", SUMMARY_JSON)
print(json.dumps(summary["standardization"], indent=2))